# Análise de posts go instagram de presidenciáveis do Brasil em 2026

<code style="color:#4a304f"><font color= #d689e6>Redes Sociais</font></code>
<code style="color:#4a304f"><font color= #d689e6>Nível 3</font></code>

&nbsp;

#### **DADOS**
São dados reais extraidos dos perfis dos 5 candidatos analisados através do Apify, no formato json, que apontam as características dos posts de cada candidato: quantidade de curtidas, data e horário, duração do vídeo, quem foi coautor etc.

&nbsp;

&nbsp;

#### **OBJETIVOS**
- Compreender os padrões de comportamento dos candidatos em suas postagens no intagram
- Padrões de comportamento de quem interagem com as postagens
- Identificar o padrão de comportamento ou conteúdo que trouxe mais benefícios para cada um

&nbsp;

&nbsp;

#### **FERRAMENTAS**
<img src="https://www.intelliswift.com/cms_images/data-ai-img/databricks/databricks-icon-logo-bnr.png" /> &nbsp;&nbsp;&nbsp; <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/cf/New_Power_BI_Logo.svg/120px-New_Power_BI_Logo.svg.png" />

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Databricks&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;PowerBI (DAX e PowerQuery)


&nbsp;

&nbsp;

#### **ETAPAS**
##### 1. Preparação dos dados para análise
##### 2. EDA
##### 3. Visualização (Power BI)

## Sobre as eleições

Algumas informações importantes que devem ser consideradas sobre os partidos e candidatos que podem contribuir para o enriquecimento dessa análise:

&nbsp;


##### Dinheiro do Estado (Recursos Públicos)
**Fundo eleitoral**: A distribuição dos recursos do Fundo Eleitoral (FEFC) para as Eleições de 2026 já foi definida pelo Tribunal Superior Eleitoral (TSE). O valor total disponível é de aproximadamente R$ 4,96 bilhões. `[foi recebido no fim de agosto]`

**Fundo Partidário**: Recurso permanente, repassado mensalmente pela União. Serve para custear despesas administrativas e de manutenção dos partidos (aluguel, pessoal, contas).

#### Dinheiro Privado
Vindo de depósito direto de pessoas físicas ao partido (incluindo os próprios candidatos) ou de financiamento coletivo.

&nbsp;


| Partido | Candidato(a) | Recursos Públicos (R$) | Recursos Privados (RS)
|---|---|---|---|
| PL | Flávio Bolsonaro | 1,94 bilhões | 139,40 milhões | 
| PT | Lula | 1,18 bilhões  | 77,60 milhões
| PSD | Ronaldo Caiado | 908,87 milhões  | 157,50 milhões
| Missão | Renan Santos | 6,59 milhões | 5,51 milhões |
| UP | Samara Martins | 4,95 milhões | 463,60 mil |

Fonte: TSE (data 28.09.2026 - 04:05) [valores arredondados]


In [0]:
%pip install seaborn

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

from functools import reduce
import math
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df1 = spark.read.option("multiLine", True).json("/Volumes/workspace/default/primeirovolume/politics2026/renan.json")
df2 = spark.read.option("multiLine", True).json("/Volumes/workspace/default/primeirovolume/politics2026/lula.json")
df3 = spark.read.option("multiLine", True).json("/Volumes/workspace/default/primeirovolume/politics2026/flavio.json")
df4 = spark.read.option("multiLine", True).json("/Volumes/workspace/default/primeirovolume/politics2026/caiado.json")
df5 = spark.read.option("multiLine", True).json("/Volumes/workspace/default/primeirovolume/politics2026/samara_m.json")

# ----- setup de mapas -----
# dataframes
dfs = {"renan": df1, "lula": df2, "flavio": df3, "ronaldo": df4, "samara": df5}

# perfis
owner_map = {"renan": "renansantosmbl", "lula": "lulaoficial", "flavio": "flaviobolsonaro", "ronaldo": "ronaldocaiado", "samara": "samaramartinsup"}

In [0]:
# Visualização geral
display(df5)
# teste de commit de outputs

In [0]:
for i, df in dfs.items():
  
  print(f'Nome:{i}')
  print(f'Linhas: {df.count()}')
  print(f'Total de Colunas: {len(df.columns)}')
  print(f'Colunas: {df.columns}')
  display(df.describe())
  print('='*150)

In [0]:

# listar colunas diferentes que tem apenas no df5 em relação ao df2
cols = []
for col in df5.columns:
  if col not in df2.columns:
     cols.append(col)
     print(col)
     content = df5.filter(df5[col].isNotNull())
     print(content.count())
     display(content[col])


# verificar nulos por colunas vazias
def verificarColunasVazias(dfs):
    print("Colunas Totalmente Vazias: ")
    colunas = []
    for i, df in dfs.items():
        print(f'-'*15)
        print(f'{i}:')
        for col in df.columns:
            if df.filter(df[col].isNotNull()).count() == 0:
                colunas.append(col)
        
        print(colunas)
        colunas = []
        
verificarColunasVazias(dfs)

## **1. Preparação dos dados**

### **1.1 Remover dados que não contribuem para os objetivos**

| Coluna | Descrição | Ação |
|---|---|---|
| `alt` | texto alternativo | ✅ |
| `audioUrl` | url do audio inserido | ⚠️ extrair informação daqui |
| `caption` | Descrição da postagem | ✅ |
| `childPosts` | imagens associadas à postagem | ⚠️ extrair informação daqui|
| `coauthorProducers` | co autores da postagem | ⚠️ extrair informação daqui|
| `commentsCount` | total de comentários | ✅  |
| `dimensionsHeight` |  altura da imagem | ❌ |
| `dimensionsWidth` | largura da imagem | ❌ |
| `displayUrl` | url da imagem | ❌ |
| `firstComment` | primeiro comentário  | ✅ |
| `hashtags` | hashtags utilizadas | ⚠️ extrair informação daqui |
| `id` | id da postagem  | ✅ |
| `images` | url das imagens | ⚠️ extrair informação daqui  |
| `inputUrl` | url da postagem | ❌ |
| `isCommentsDisabled` | se os comentários estão desabilitados | ✅ |
| `isPinned` | se o post foi fixado | ❌ |
| `latestComments` | os comentários mais recentes | ⚠️ extrair informação daqui |
| `likesCount` | total de likes | ✅ |
| `locationId` | id da localização | ✅|
| `locationName` | nome da localização | ❌ |
| `mentions` | postaggens onde um perfil foi marcado | ✅ |
| `musicInfo` | informações da música | ✅ extrair informação daqui |
| `originalHeight` | altura original da imagem | ❌ |
| `originalWidth` | largura original da imagem | ❌ |
| `ownerFullName` | nome completo do user| ❌ |
| `ownerId` | id do ig | ❌ |
| `ownerUsername` | username do autor da postagem | ✅ extrair informação daqui |
| `paidPartnership` | se foi parceria paga | ✅ |
| `productType` | tipo de postagem | ✅ |
| `shortCode` | código curto da url da postagem | ❌ |
| `taggedUsers` | perfis marcados | ✅ extrair informação daqui |
| `timestamp` | timestamp no qual foi postado | ✅ |
| `type` | formato do conteúdo | ✅ |
| `url` | url  | ❌ |
| `videoDuration` | duração do vídeo | ✅ |
| `videoPlayCount` | contagem de plays no vídeo | ✅ |
| `videoUrl` | url do vídeo | ❌ |
| `videoViewCount` | visualizações do vídeo | ✅ |


além dessas colunas, o scrapper pegou +13 colunas apenas no json de Samara e elas tem apenas 2 linhas de conteúdo, serão dropadas por não apresentarem nada relevante além de dificultarem a comparação.


In [0]:
# ----- FUNÇÕES DE APOIO PARA DROP ----- #
# alinhas as colunas em comum dos dataframes e dropar as excedentes
def alinharIntersecao(dfs):
    conjuntos = [set(df.columns) for df in dfs.values()]
    colunas_comuns = set.intersection(*conjuntos)
    
    primeiro = next(iter(dfs.values()))
    ordem_ref = [c for c in primeiro.columns if c in colunas_comuns]
    
    dfs = { nome: df.select(*ordem_ref) for nome, df in dfs.items() }

    return dfs

# Dropar colunas desnecessárias
def droparColunasIniciais(dfs):
    colunas = ['dimensionsHeight', 'locationName', 'dimensionsWidth', 'displayUrl', 'inputUrl', 'originalHeight', 'originalWidth', 'ownerFullName', 'shortCode', 'url', 'videoUrl', 'isPinned']
    for nome, df in dfs.items():
        dfs[nome] = df.drop(*colunas)



In [0]:

dfs = alinharIntersecao(dfs)

droparColunasIniciais(dfs)

### **1.2 Colunas que precisam de um tratamento mais meticuloso**

| Coluna | Descrição | Ação |
|---|---|---|
| `audioUrl` | url do audio inserido | ⚠️ extrair informação daqui |
| `childPosts` | imagens associadas à postagem | ⚠️ extrair informação daqui|
| `coauthorProducers` | co autores da postagem | ⚠️ extrair informação daqui|
| `hashtags` | hashtags utilizadas | ⚠️ extrair informação daqui |
| `images` | url das imagens | ⚠️ extrair informação daqui  |
| `latestComments` | os comentários mais recentes | ⚠️ extrair informação daqui |

#### 💠 **ChildPosts x Images**

`Child posts `são as imagens associadas à postagem, mas não parecem trazer muita informação de forma separada. Cada `Child Post` contém todas as informações de uma postagem normal porém tem a limitação de serem apenas derivadas da postagem original. Por exemplo: O número de likes de um `child post` é o mesmo que da postagem na qual ele está contido.

O que pode ser diferente? **`Tagged Users`**, pode ser que cada imagem tenha pessoas diferentes "marcadas" alé das informações de id e url próprios do childpost.

Considerando o que precisa o objetivo, poderia ser interessante saber se os responsáveis pela conta marcaam pessoas ao longo da postagem e se o post original contem todas as pessoas marcadas nos `child posts `nas sua própria informação de `tagged users`.

Se quisermos apenas contar quantos imagens contem o post, bastaria fazer a contagem de `images`.

Nesse caso, o caminho que vamos traçar é:

- ###### Comparar a quantidade de `tagged users` nos `child posts` x `post principal`
- ###### Comparar a quantidade de `images` x `ChildPost`
- ###### Comparar a quantidade de carrosseis x `Child posts`

#### 💠 **CoauthorProducers**

Assim como `Child Posts`, `CoauthorProducers` tem seus próprios atribudos, esses sendo diferentes de um para outro e, por isso, requerem mais atenção.
o que importa:
- Quantos co-autores essa postagem tem? `len()`
- Quem são esses co-autores? `username`
- Esse perfil é verificado? `is_verified`

#### 💠 **Hashtags**

São poucas e não trazem muita informação, podemos manter elas como um array e pegar o tamanho dele para separar uma contagem.

#### 💠 **Audio URL**

Serve apenas para saber se essa postagem tem ou não música.



In [0]:
# ----- FUNÇÕES DE APOIO PARA ANÁLISE DE TAGGED USERS E CHILD POSTS ----- #

# Comparar tagged Users child x original
def compararTaggedUser(dfs):
    for i, df in dfs.items():
        comparacaoTaggedUsers = 0
        print(f'{i}:')
        
        child_df = df.where("SIZE(childPosts) > 0") \
            .select(explode(df['childPosts']).alias('child'))
            
        taggedUsersChild = child_df.filter(size(child_df['child.taggedUsers']) > 0) \
            .select(explode(child_df['child.taggedUsers'])).count()

        taggedUsersOriginal = df.filter(size(df['childPosts']) > 0) \
            .filter(size(df['taggedUsers']) > 0) \
            .select(explode(df['taggedUsers'])).count()
      
        totalTaggedUsers = df.where("SIZE(taggedUsers) > 0") \
            .select(explode(df['taggedUsers'])).count()

        comparacaoTaggedUsers = taggedUsersOriginal - taggedUsersChild
        print(f'Comparação Tagged Users Original - Child: {comparacaoTaggedUsers} \n Child [{taggedUsersChild}] Original [{taggedUsersOriginal}] Todos os posts [{totalTaggedUsers}]')


# comparar relacao carrossel x child posts
def compararRelacaoChildPostxCarrosselxImagem(dfs):
    for i, df in dfs.items():
        qtdCarrOriginal = df.filter(size(df['childPosts']) > 0) \
            .filter(df['productType'].contains('carousel_container')).count()
        
        hasChildPost = df.filter(size(df['childPosts']) > 0).count()

        qtdCarrOriginal = df.filter(size(df['childPosts']) > 0) \
            .filter(df['productType'].contains('carousel_container')).count()
            
        qtdImgChildPosts = df.filter(size(df['childPosts']) > 0) \
            .select(explode(df['childPosts'])).count()

        qtdImgOriginal = df.filter(size(df['childPosts']) > 0) \
                .select(explode(df['images'])).count()

        comparacaoRelacaoCarr = qtdCarrOriginal - hasChildPost
        comparacaoRelacaoImg = qtdImgOriginal - qtdImgChildPosts
        print(f'{i}:')
        print(f'direrença child x carrossel: {comparacaoRelacaoCarr} \n Original [{qtdCarrOriginal}] Child Posts [{hasChildPost}]\n')
        print(f'direrença child x img: {comparacaoRelacaoImg} \n Original [{qtdImgOriginal}] Child Posts [{qtdImgChildPosts}]')
        print('='*150)



In [0]:

compararTaggedUser(dfs)

print('='*150)

compararRelacaoChildPostxCarrosselxImagem(dfs)


#### 💠 **Análise de ChildPosts x Images**

O dataframe samara teve um problema de erro em 2 linhas, o problema pareceu envolver usuário privado com base nas 13 colunas que foram criadas, sendo essa umas das limitações do scrapping. Ter esse número reaparecendo como erro nos dois testes acima pode indicar que, talvez, esse mesmo erro esteja afetando a relação Child post x imagens x carrosel. Com exceção de samara, a quantidade de imagens representou exatamente a quantidade de child post da postagem feita. Podemos assumir contar a **quantidade de imagens da postagem campo imagens é seguro**, assim como, **todos os child posts fazem parte de carroseis**.

Comparação Tagged Users Original x Child:

-  Valores muito próximos ou iguais
- o valor de samara (-2) pode indicar relação com o erro identificado no começo
- Ronaldo tem uma difereça maior que os outros (-12)
- o child, quando não é examente igual ao original, é sempre maior.

`avaliação: a chance dos dois números representarem o mesmo valor é muito grande, mas entender a ogirem dessa diferença também é importante.` 
- `[ Hipótese 1]` Existem marcações repetidas em diferentes carrosseis e o taggedUser não inclui repetições em sua listagem
- `[ Hipótese 2]` Inconsistência no scrapping

como 12 é um número proporcionamente grande de erro dentro do dataframe Ronaldo, vale a pena investigar essa hipótese

In [0]:
from pyspark.sql.functions import col

def montarListaUsers(df):
    child_list = df.where("SIZE(childPosts) > 0") \
            .select(df['id'].alias('id_original'), explode(df['childPosts']).alias('child'))

    taggedU = child_list.where("SIZE(child.taggedUsers) > 0") \
        .select(child_list['id_original'],explode(child_list['child.taggedUsers.username']).alias('username'))
    
    return taggedU



def verificarTaggedUserRepetido(dfs):
    for i, df in dfs.items():
        TUrepetido = 0
        taggedU = montarListaUsers(df)     

        repeticoes = taggedU.groupBy('id_original', 'username').count() \
            .filter('count > 1')
        
        resultado = repeticoes.agg(sum(col('count') - 1).alias('total_rep')).collect()[0]
        TUrepetido = resultado['total_rep'] if resultado['total_rep'] is not None else 0
        
        print(f'{i}: Repetições entre child posts do mesmo post: {TUrepetido}')


verificarTaggedUserRepetido(dfs)



##### **Conclusão**

A incidência de repetições se equipara à diferenças de quantidade de pessoas marcadas em postagens e isso confirma a hipótese número 1. Agora com a compreensão total dessas informações, o caminho a ser feito será:
- Contar fotos do carrossel pela coluna imagem
- Gerar colunas taggedUsername com uma lista de usernames distintos marcados em cada postagem e excluir a coluna childPosts
- Usar a coluna taggedUser como referência para a quantidade de pessoas marcadas

In [0]:
# funções de transformação para Child Post
def addColunaTaggedUsernames(dfs):
    for i, df in dfs.items():
        usernames_por_id = montarListaUsers(df).groupBy('id_original') \
            .agg(collect_set('username').alias('taggedUsernames'))

        df = df.join(usernames_por_id, df['id'] == usernames_por_id['id_original'], 'left') \
            .drop('id_original') \
            .drop('childPosts')

        dfs[i] = df

# total imagens por carrossel
def contarImagensPorCarrossel(dfs):
    for i, df in dfs.items():
        df = df.withColumn('totalImagens', size(df['images']))
        df = df.drop('images')
        dfs[i] = df

In [0]:
addColunaTaggedUsernames(dfs)

contarImagensPorCarrossel(dfs)

####💠 **Criador original da Postagem**

Com a ferramenta de coautoria do instagram, nem todos os posts do perfil são originalmente vindos deles, por isso é necessária a criação de uma coluna perfil, para identificar qual candidato(a) estamos analisando, e se o post é de autoria própria ou não.

In [0]:
# owner_map: {perfil: ownerUsername}
def marcar_owner(df, owner_map):
    expressao = None
    for perfil, owner_esperado in owner_map.items():
        condicao = (col("perfil") == perfil) & (col("ownerUsername") == owner_esperado)
        
        expressao = when(condicao, lit(True)) if expressao is None \
                    else expressao.when(condicao, lit(True))
    
    df = df.withColumn("is_owner", expressao.otherwise(lit(False)))
    return df

def unificarDataframes(dfs):
    df_completo = []
    for df in dfs.values():
        # Alinhar schema: remover fbid de coauthorProducers se existir
        ''' if 'fbid' in df.schema['coauthorProducers'].dataType.elementType.fieldNames():
            df = df.withColumn('coauthorProducers',
                transform(col('coauthorProducers'), lambda x: x.dropFields('fbid')))'''
        df_completo.append(df)
    return reduce(lambda a, b: a.unionByName(b), df_completo)
    
def associarPerfilaoPost(dfs):
   return {
        nome: df.withColumn("perfil", lit(nome))
        for nome, df in dfs.items()
    }
   

    


In [0]:

dfs = associarPerfilaoPost(dfs)

# Normalizar coauthorProducers: remover campo fbid (presente apenas no perfil samara)
for nome, df_temp in dfs.items():
    if 'fbid' in df_temp.schema['coauthorProducers'].dataType.elementType.fieldNames():
        dfs[nome] = df_temp.withColumn('coauthorProducers',
            transform(col('coauthorProducers'), lambda x: x.dropFields('fbid')))

df = unificarDataframes(dfs)

df = marcar_owner(df, owner_map)

df = df.withColumn("videoDuration", col("videoDuration").cast(DoubleType()))

df.display()



#### 💠 **Análise de Coautores, Hashtag e músicas**

Coautores tem esse formato:

![image_1791282256861.png](./image_1791282256861.png "image_1791282256861.png")

daqui, queremos manter as features:
- username, isVerified (qtd)

para hashtags, vamos:
- contar a quantidade



In [0]:

# Aplica dropFields apenas quando perfil == "samara"
df = df.withColumn(
    "coauthorProducers",
    when(
        col("perfil") == "samara",
        transform(col("coauthorProducers"), lambda x: x.dropFields("fbid"))
    ).otherwise(col("coauthorProducers"))
)

# Lista de usernames dos coautores
df = df.withColumn(
    "lista_coautores",
    coalesce(
        transform(col("coauthorProducers"), lambda x: x.username),
        lit([]).cast("array<string>")
    )
)

# Quantidade de coautores
df = df.withColumn(
    "qtd_coautores",
    coalesce(size("coauthorProducers"), lit(0))
)

# Quantidade de coautores verificados
df = df.withColumn(
    "qtd_coautores_verificados",
    coalesce(
        size(filter("coauthorProducers", lambda x: x.is_verified == True)),
        lit(0)
    )
)

# Quantidade de hashtags
df = df.withColumn(
    "qtd_hashtags",
    coalesce(size("hashtags"), lit(0))
)

# música com audio
df = df.withColumn(
    "post_com_musica",
    when(
        col("musicInfo").isNotNull() & col("musicInfo").isNotNull(),
        True
    ).otherwise(False)
)

# dropar colunas desnecessárias
df = df.drop('coauthorProducers', 'taggedUsers', 'audioUrl', 'musicInfo')

df.display()

#### 💠 **Comentários**

Comentários em si tem forma de análise diferente e mais aprofundada do que a proposta desse estudo. Por hora, vamos separar em uma novo dataframe mantendo aquilo que é essencial.

A estura é a seguinte:

![image_1791284039947.png](./image_1791284039947.png "image_1791284039947.png")

vamos criar dois dataframes um com usuário, e outro com os comentarios.

o Usuário terá:
- id
- username
- is_mentionable
- is_private
- is_verified

O comentário terá:
- id_post
- id
- ownerusername
- text
- repliesCount
- likesCount
- TimeStamp


In [0]:
''' extrair de latestComents para um novo dataframe: 
    id_post
    id
    ownerusername
    text
    repliesCount
    likesCount
    TimeStamp '''

def extrairLatestComments(df):
    latestComments = df.select(
        col('id').alias('id_post'),
        explode('latestComments').alias('latestComment')
    ).select(
        col('id_post'),
        col('latestComment.id').alias('id'),
        col('latestComment.owner.username').alias('ownerusername'),
        col('latestComment.text').alias('text'),
        col('latestComment.repliesCount').alias('repliesCount'),
        col('latestComment.likesCount').alias('likesCount'),
        col('latestComment.timestamp').alias('timestamp')
    )
    return latestComments

latestComments = extrairLatestComments(df)
latestComments.display()





In [0]:
''' extrair de latestComents.owner para um novo dataframe: 
    id
    username
    is_mentionable
    is_private
    is_verified'''

def extrairLatestCommentsOwner(df):
    latestCommentsOwner = df.select(
        col('id').alias('id_post'),
        explode('latestComments.owner').alias('user')
    ).select(
        col('id_post'),
        col('user.id').alias('id'),
        col('user.username').alias('username'),
        col('user.is_mentionable').alias('is_mentionable'),
        col('user.is_private').alias('is_private'),
        col('user.is_verified').alias('is_verified')
    )
    return latestCommentsOwner

latestCommentsUser = extrairLatestCommentsOwner(df)
latestCommentsUser.display()

In [0]:
df = df.drop('latestComments')

#### 💠 **Janela de tempo**

Para finalizar antes de ir para a EDA, será necessário alinhas a janela de tempo analisada. Por questões de limitações com o Apfy, vou usar a janela de 10 de maio a 19 de setembro, exatas 19 semanas

In [0]:
df = df.filter("timeStamp BETWEEN '2026-02-01' AND '2026-10-04'")

## **2. EDA**

Vou de dividir a análise em 5 etapas:
1. Visão geral
2. Valores numéricos
3. Valores textuais
4. Valores booleanos
5. Criação de novas features

### **2.1 Visão Geral**


In [0]:
# Contagem de nulos por colunas
nulos_por_coluna = df.select(*(count(when(col(c).isNull(), c)).alias(c) for c in df.columns))
nulos_por_coluna.display()


In [0]:
# selecionar linhas com productType nulo
prodType_nulo = df.filter(col("productType").isNull())
prodType_nulo.display()


In [0]:
''' linha 1 é um post estático (img = 0) enquanto a linha 2 é um carrosel ( img = 10) '''

# setar id 3936714299352189410 como Sidecar(type) e carousel_container(productType)
df = df.withColumn("productType", when(col("id") == "3936714299352189410", "carousel_container").otherwise(col("productType")))
df = df.withColumn("type", when(col("id") == "3936714299352189410", "Sidecar").otherwise(col("type")))

# setar id 3960178502392405303 como Image(type) e feed(productType)
df = df.withColumn("productType", when(col("id") == "3960178502392405303", "feed").otherwise(col("productType")))
df = df.withColumn("type", when(col("id") == "3960178502392405303", "Image").otherwise(col("type")))

prodType_nulo = df.filter(col("productType").isNull())
prodType_nulo.display()

# Limpeza poz identificação de nulos
df = df.drop("locationId")

In [0]:
campos_numericos = ['commentsCount', 'likesCount', 'videoPlayCount', 'videoViewCount', 'qtd_coautores', 'qtd_coautores_verificados','qtd_hashtags','totalimagens', 'videoDuration']

campos_textuais = ['alt', 'caption', 'ownerUsername', 'firstcomment', 'perfil', 'type', 'productType']

campos_booleanos = ['isCommentsDisabled', 'paidPartnership', 'post_com_musica', 'is_owner']

# Aplicar fillna apenas em colunas existentes
df = df.fillna(0, subset=campos_numericos)
df = df.fillna('', subset=[c for c in ['alt', 'caption', 'ownerUsername', 'firstComment'] if c in df.columns])
df = df.fillna(False, subset=[c for c in ['isCommentsDisabled', 'paidPartnership', 'post_com_musica'] if c in df.columns])
df = df.fillna(False, subset=campos_booleanos)



In [0]:
# ============================================================
# 1. PREPARAÇÃO DOS DADOS
# ============================================================

df_pd = df.select(
    "perfil",
    "productType",
    "likesCount",
    "commentsCount",
    "videoDuration",
    "totalImagens",
    "qtd_coautores",
    "is_owner"
).toPandas()

perfis = sorted(df_pd["perfil"].dropna().unique().tolist())

# ============================================================
# 2. DEFINIÇÕES AUXILIARES
# ============================================================

# Post estático = feed + carousel_container (carrossel)
df_pd["is_video"] = df_pd["productType"] == "clips"
df_pd["is_estatico"] = df_pd["productType"].isin(["carousel_container", "feed"])

# ============================================================
# 3. CÁLCULO DAS MÉTRICAS
# ============================================================

# --- 3.1 Total de posts, likes e comentários ---
metricas_gerais = df_pd.groupby("perfil").agg(
    total_posts=("perfil", "count"),
    total_likes=("likesCount", "sum"),
    total_comments=("commentsCount", "sum")
).reindex(perfis).reset_index()

# --- 3.2 Duração de vídeos (média e mediana) ---
videos = df_pd[df_pd["is_video"]].dropna(subset=["videoDuration"])
duracao = videos.groupby("perfil")["videoDuration"].agg(
    media="mean", mediana="median"
).reindex(perfis).reset_index()

# --- 3.3 Vídeo vs estático ---
tipo_post = df_pd.groupby("perfil").agg(
    video=("is_video", "sum"),
    estatico=("is_estatico", "sum")
).reindex(perfis).reset_index()

# --- 3.4 Média e mediana de imagens por carrossel ---
carrosseis = df_pd[df_pd["productType"] == "carousel_container"]
imgs_carrossel = carrosseis.groupby("perfil")["totalImagens"].agg(
    media="mean", mediana="median"
).reindex(perfis).reset_index()

# --- 3.5 Posts com coautores ---
coautoria = df_pd.groupby("perfil").agg(
    com_coautor=("qtd_coautores", lambda x: (x > 0).sum()),
    sem_coautor=("qtd_coautores", lambda x: (x == 0).sum())
).reindex(perfis).reset_index()

# --- 3.6 Autoral vs não autoral ---
autoral = df_pd.groupby("perfil").agg(
    autoral=("is_owner", "sum"),
    nao_autoral=("is_owner", lambda x: (~x).sum())
).reindex(perfis).reset_index()

# ============================================================
# 4. GRID DE 6 GRÁFICOS (3 linhas × 2 colunas)
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(18, 20))
axes = axes.flatten()

# ---------- Gráfico 1: Total de posts, likes e comentários ----------
ax = axes[0]
x = np.arange(len(perfis))
largura = 0.28

ax.bar(x - largura, metricas_gerais["total_posts"], largura, label="Posts", color="#4C72B0")
ax.bar(x, metricas_gerais["total_likes"], largura, label="Likes", color="#DD8452")
ax.bar(x + largura, metricas_gerais["total_comments"], largura, label="Comentários", color="#55A868")

# Rótulos
for i, (p, l, c) in enumerate(zip(metricas_gerais["total_posts"],
                                   metricas_gerais["total_likes"],
                                   metricas_gerais["total_comments"])):
    ax.text(i - largura, p, f"{int(p):,}", ha="center", va="bottom", fontsize=8)
    ax.text(i, l, f"{int(l):,}", ha="center", va="bottom", fontsize=8)
    ax.text(i + largura, c, f"{int(c):,}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(perfis, rotation=45)
ax.set_title("Total de posts, likes e comentários por candidato")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

# ---------- Gráfico 2: Duração de vídeos (média e mediana) ----------
ax = axes[1]
x = np.arange(len(perfis))

ax.bar(x - 0.2, duracao["media"], 0.4, label="Média", color="#4C72B0")
ax.bar(x + 0.2, duracao["mediana"], 0.4, label="Mediana", color="#DD8452")

for i, (m, med) in enumerate(zip(duracao["media"], duracao["mediana"])):
    if not pd.isna(m):
        ax.text(i - 0.2, m, f"{m:.1f}s", ha="center", va="bottom", fontsize=8)
    if not pd.isna(med):
        ax.text(i + 0.2, med, f"{med:.1f}s", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(perfis, rotation=45)
ax.set_title("Tempo médio e mediano da duração dos vídeos (s)")
ax.set_ylabel("Duração (segundos)")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

# ---------- Gráfico 3: Vídeo vs Estático (pizza) ----------
ax = axes[2]
total_geral = tipo_post[["video", "estatico"]].sum()
ax.pie(
    total_geral,
    labels=["Vídeo", "Estático (Feed + Carrossel)"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#4C72B0", "#DD8452"]
)
ax.set_title("Total de vídeo vs estático (todos os perfis)")

# ---------- Gráfico 4: Imagens por carrossel (média e mediana) ----------
ax = axes[3]
x = np.arange(len(perfis))

ax.bar(x - 0.2, imgs_carrossel["media"], 0.4, label="Média", color="#4C72B0")
ax.bar(x + 0.2, imgs_carrossel["mediana"], 0.4, label="Mediana", color="#DD8452")

for i, (m, med) in enumerate(zip(imgs_carrossel["media"], imgs_carrossel["mediana"])):
    if not pd.isna(m):
        ax.text(i - 0.2, m, f"{m:.1f}", ha="center", va="bottom", fontsize=8)
    if not pd.isna(med):
        ax.text(i + 0.2, med, f"{med:.1f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(perfis, rotation=45)
ax.set_title("Média e mediana de imagens por carrossel")
ax.set_ylabel("Quantidade de imagens")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

# ---------- Gráfico 5: Coautoria (pizza) ----------
ax = axes[4]
total_coaut = coautoria[["com_coautor", "sem_coautor"]].sum()
ax.pie(
    total_coaut,
    labels=["Com coautor", "Sem coautor"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#55A868", "#C44E52"]
)
ax.set_title("Percentual de postagens com coautores (todos os perfis)")

# ---------- Gráfico 6: Autoral vs não autoral ----------
ax = axes[5]
x = np.arange(len(perfis))
largura = 0.4

ax.bar(x - largura/2, autoral["autoral"], largura, label="Autoral", color="#55A868")
ax.bar(x + largura/2, autoral["nao_autoral"], largura, label="Não autoral", color="#C44E52")

for i, (a, n) in enumerate(zip(autoral["autoral"], autoral["nao_autoral"])):
    ax.text(i - largura/2, a, f"{int(a):,}", ha="center", va="bottom", fontsize=8)
    ax.text(i + largura/2, n, f"{int(n):,}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(perfis, rotation=45)
ax.set_title("Postagens autorais vs não autorais por candidato")
ax.set_ylabel("Quantidade de posts")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

# ---------- Ajuste final ----------
plt.suptitle("Análise comparativa por candidato", fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

#### Primeiras impressões:
1. Quantidades de interações com perfis é massivamente diferente.
Flávio, Lula e Renan estão num patamar numéricos de likes e comentários muito diferente de Ronaldo e Samara, sendo que a candidato com mais curtidas no período (flávio) tem 48,2 vezes mais a quantidade de samara (a que tem menos curtidas). Apesar disso, flávio foi um dos candidatos com menos conteúdos publicados enquanto samara foi uma das que mais investiu em publicações.

2. Quanto à duração dos vídeos:
Renan tendem a ter vídeos mais longos en quanto Ronaldo tem vídeos mais curtos em termos de média e mediana, mas todas as medianas estão abaixo de 120s de duração, o que mostra o entendimento dos candidatos do tipo de vídeo que melhor funcionam para essa rede.

3. Vídeo é o tipo de conteúdo predominante
70% dos postas analisados são vídeos, isso mostra o entendimento dos candidatos de que vídeos são importantes para apresentar suas candidaturas dentro do instagram. É uma forma de mostrar um pouco da pessoa real no digital, por exemplo, mas pode ser usado de várias formas: construção de narrativas audiovisuais, apresentação de denúncias, mostar o candidato com o povo etc.

4. A maioria das postagens são propriedade dos candidatos, mas Samara segue o fluxo contrário
Curiosamente, a candidata Samara faz exatamente o oposto de todos os outros candidatos e vale a pena investigar o porque no aprofundamento da EDA.

### 2.2 Campos Numéricos

In [0]:
# analisar campos numéricos
display(df.select(*campos_numericos).describe())

In [0]:
# postagens com likesCount menores que 
df.filter(df.likesCount < 0).display()

#### 2.2.1 Entendendo o caso de -1 like

Vamos analisar algumas postagens diretamente na fonte:

1. **Vídeo no qual Lula é coautor**
 info | valor | obs |
|---|---|---|
| likes | 15 | no app mostra só o coração acinzentado
| comentários | 15| comentários restritos
| republicações | 4290|
| compartilhamentos | 6003|
| comentário mais curtido | 1486 likes|
| comentário menos curtido | 105 likes|

2. **Post estático original de ronaldo**
 info | valor | obs |
|---|---|---|
| likes | 20 | no app mostra só o coração acinzentado
| comentarios | 20| 
| republicações | 25|
| compartilhamentos |2|
| comentario mais curtido | 3 likes|
| comentario menos curtido | 0 likes|

3. **Vídeo no qual Samara é coautora**
 info | valor | obs |
|---|---|---|
| likes | ? | no app mostra só o coração acinzentado
| comentarios | 8| 
| republicações | 237|
| compartilhamentos |0|
| comentario mais curtido | 1 likes|
| comentario menos curtido | 0 likes|

O que podemos entender daqui:

Com a análise analógica, foi notado que essas postagens tem o botão de like comprometido, provavelmente os likes foram ocultados pelos autores ou pelo instagram por algum motivo e por isso está -1. Daqui temos 2 caminhos: 
- identificar esses posts como "likes ocultos"
- estimar com base em performance, o que esses posts podem ter em quantidades de likes.

Como temos poucos parâmetros para fazer uma extimativa tão complexa, vamos trabalhar com a segunda opção e excluir esses posts das análises com like


In [0]:
# feature LikesCount desconlhecido quando LikesCount for -1 (True)
df = df.withColumn("LikesCount_unknown", when(df.likesCount == -1, True).otherwise(False))


#### 2.2.2 Distribuição Geral

In [0]:
# subgrupos para pairplot

# likes, comentários, ,videoPlayCount, videoViewCount, videoDuration
sub_performance = [
    "likesCount",
    "commentsCount",
    "videoPlayCount",
    "videoViewCount",
]

sub_comportamento = [
   'qtd_coautores',
   'qtd_coautores_verificados',
   "videoDuration",
   'totalimagens'
   ]

sub_comp_perform = [
   'qtd_coautores',
   "videoDuration",
   'totalimagens',
    "likesCount",
    "videoViewCount",
    "commentsCount"
   ]

In [0]:
# Pairplot de performance
df_pd = df.filter("likesCount > 0").select("perfil", *sub_performance).toPandas()

sns.pairplot(
    df_pd,
    hue="perfil",           
    diag_kind="kde",        
    plot_kws={"alpha": 0.5, "s": 15},  
    corner=False            
)
plt.suptitle("Distribuição de performance por perfil", y=1.02, fontsize=14)
plt.show()

# Pairplot de comportamento
df_pd = df.select("perfil", *sub_comportamento).toPandas()

sns.pairplot(
    df_pd,
    hue="perfil",           
    diag_kind="kde",        
    plot_kws={"alpha": 0.5, "s": 15},  
    corner=False            
)
plt.suptitle("Distribuição de comportamento por perfil", y=1.02, fontsize=14)
plt.show()

# Pairplot de comportamento x performance
df_pd = df.filter("likesCount > 0").select("perfil", *sub_comp_perform).toPandas()

sns.pairplot(
    df_pd,
    hue="perfil",           
    diag_kind="kde",        
    plot_kws={"alpha": 0.5, "s": 15},  
    corner=False            
)
plt.suptitle("Distribuição de comportamento x performance por perfil", y=1.02, fontsize=14)
plt.show()

**O que os pair plots nos dizem**

1. Não existe forte relação entre a quantidade de imagems do carrosel e a performance do post
2. Samara é a única candidata que tem um volume consideravel de postagens com 2 à 3 e 4 coautores.
3. Renan foca mais em videos um pouco mais longos em relaçção aos outros candidatos. Não é apenas um pico de duração, é uma decisão.
4. Lula possiu pocuos videos com uma duração muito além do padrão dos outros candidatos.
5. Não existe relação entre a quantidade de coautores e a performance das postagens
6. VideoViewsCounts e VideosPlayCount provavelmente tem alguma relação de derivação

In [0]:
# Trazer os dados necessários para o Pandas
df_pd = df.select("perfil", "likesCount", "commentsCount", "videoViewCount").toPandas()

# Remover valores inválidos (-1 de likes ocultos)
df_pd = df_pd[df_pd["likesCount"] >= 0]

# Ordem dos perfis
ordem_perfis = sorted(df_pd["perfil"].dropna().unique().tolist())

fig, axes = plt.subplots(3, 1, figsize=(12, 15))

campos = [
    ("likesCount", "Likes por perfil"),
    ("commentsCount", "Comentários por perfil"),
    ("videoViewCount", "Views de vídeo por perfil"),
]

# Plot
for ax, (campo, titulo) in zip(axes, campos):
    sns.boxplot(
        data=df_pd,
        x="perfil",
        y=campo,
        order=ordem_perfis,
        palette="Set2",
        hue="perfil",
        ax=ax
    )
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.suptitle("Comparação de engajamento por perfil", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**O que os box plots nos dizem**

1. Apesar de flávio ter mais consistência no grande volume de likes, views em vídeo e comentários, o vídeo com mais views foi de Lula
2. Renan tem uma performance de views em video melhor que a de Lula, mas não tem os seus vídeos dentre os de melhor performance.
3. o Vídeo com mais views de Samara, tem a performance dentro do quadrante mediano nos videos de Flavio.
4. Os Vides com mais views de Ronaldo vão muito além da sua performance mediana e ele tem vídeos com melhor performance que os de Samara.
5. Renan tem uma consistencia proporcianal em performances de videos, likes e comentários. Não possui outliers extremos com é no caso de Lula, Flávio e Ronaldo.

#### 2.2.3 Período de tempo

In [0]:
# Converter timestamp de string para timestamp e truncar por mês
df_tempo = (
    df.withColumn("ts", to_timestamp("timestamp"))
      .withColumn("mes", date_trunc("month", col("ts")))
)

# Contar posts por (perfil, mês)
posts_por_mes = (
    df_tempo.groupBy("perfil", "mes")
            .agg(count("*").alias("qtd_posts"))
            .orderBy("mes", "perfil")
)

# Trazer para Pandas 
posts_pd = posts_por_mes.toPandas()
posts_pd["mes"] = posts_pd["mes"].dt.strftime("%Y-%m") 
posts_pd = posts_pd.sort_values("mes")

# Plot
plt.figure(figsize=(14, 6))
sns.lineplot(
    data=posts_pd,
    x="mes",
    y="qtd_posts",
    hue="perfil",
    marker="o",
    linewidth=2
)

plt.title("Distribuição de posts por candidato ao longo do tempo")
plt.xlabel("Mês")
plt.ylabel("Quantidade de posts")
plt.xticks(rotation=45)
plt.legend(title="Perfil", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [0]:
# Preparar dados no Spark
df_semana = (
    df.withColumn("ts", to_timestamp("timestamp"))
      .withColumn("semana", date_trunc("week", col("ts")))
      .filter(col("likesCount") >= 0)
)

agregadoL = (
    df_semana.groupBy("perfil", "semana")
             .agg(
                 count("*").alias("qtd_posts"),
                 avg("likesCount").alias("media_likes"),
                 expr("percentile_approx(likesCount, 0.5)").alias("mediana_likes")
             )
             .orderBy("semana", "perfil")
)

# Trazer para Pandas
df_pd = agregadoL.toPandas()
df_pd1 = agregadoL.toPandas()
df_pd["semana"] = df_pd["semana"].dt.strftime("%Y-%m-%d")
df_pd = df_pd.sort_values("semana")

# Ordem dos perfis
ordem_perfis = sorted(df_pd["perfil"].dropna().unique().tolist())

# Plot
plt.figure(figsize=(16, 7))
ax = sns.lineplot(
    data=df_pd,
    x="semana",
    y="mediana_likes",      
    hue="perfil",
    hue_order=ordem_perfis,
    marker="o",
    linewidth=2
)

plt.title("Mediana de likes por semana (número = qtd de posts)", fontsize=13)
plt.xlabel("Semana")
plt.ylabel("Mediana de likes")
plt.xticks(rotation=45)
plt.legend(title="Perfil", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [0]:
# Filtrar só vídeos e preparar
df_video = (
    df.withColumn("ts", to_timestamp("timestamp"))
      .withColumn("semana", date_trunc("week", col("ts")))
      .filter(col("videoDuration") > 0)  
)

agregadoV = (
    df_video.groupBy("perfil", "semana")
            .agg(
                count("*").alias("qtd_videos"),
                avg("videoDuration").alias("media_duracao"),
                expr("percentile_approx(videoDuration, 0.5)").alias("mediana_duracao")
            )
            .orderBy("semana", "perfil")
)

# Pandas
df_pd = agregadoV.toPandas()
df_pd["semana"] = df_pd["semana"].dt.strftime("%Y-%m-%d")
df_pd = df_pd.sort_values("semana")

ordem_perfis = sorted(df_pd["perfil"].dropna().unique().tolist())

# Plot
plt.figure(figsize=(16, 7))
ax = sns.lineplot(
    data=df_pd,
    x="semana",
    y="mediana_duracao",   
    hue="perfil",
    hue_order=ordem_perfis,
    marker="o",
    linewidth=2
)

plt.title("Mediana de duração dos vídeos por semana (número = qtd de vídeos)", fontsize=13)
plt.xlabel("Semana")
plt.ylabel("Duração (segundos)")
plt.xticks(rotation=45)
plt.legend(title="Perfil", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

1. Intensificação de quantidade de postagens a partir de Agosto.
Com exceção de flávio, todos os candidatos tiveram um aumento na frequênciad e conteúdo de julho para agosto, mês no qual a corrida eleitoral começou.

2. Flavio e Lula investeram a mediana dos likes no pefil.
Antes da Campanha, lula tinha uma mediana maior de likes e eqnautno Flávio estava abaixo, no mês de início da campanha, a mediana de Flávio subiu muito enquanto a de Lula se mostrou crescente, porém baixa em relação à pré campanha.

2. Estratégia com vídeos.
Existe uma leve tendência de diminuição mediana da druação dos vídeos entre boa parte dos vandidatos, esm especial,  entre os que dinham pediana mais alta na pré campanha.

### **2.3 Campos Booleanos**

In [0]:
# Analise de booleanos

# gráfico de pizza para comparar true x false 
df_pd = df.select(*campos_booleanos).toPandas()

fig, axes = plt.subplots(1, len(campos_booleanos), figsize=(15, 5))
axes = axes.flatten()

for i, campo in enumerate(campos_booleanos):
    ax = axes[i]
    dados = df_pd[campo].dropna()
    
    if dados.empty:
        ax.set_title(f'{campo} (sem dados)')
        ax.axis('off')
        continue
    
    sns.countplot(x=dados, ax=ax)
    ax.set_title(f'{campo} — True vs False')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

In [0]:
#drop de campos sem contribuição relevante para a análise
df = df.drop("isCommentsDisabled", "paidPartnership")

### 2.4 Campos Textuais

In [0]:
#visão geral campos de texto

df_text = df.select(*campos_textuais).toPandas()
df_text.describe()


In [0]:
# analisar campos textuais
# COAUTORES
explodido = df.select(col("perfil"), explode("lista_coautores").alias("coautor"))
contagem = (explodido.groupBy("perfil", "coautor").agg(count("*").alias("qtd")))
window = Window.partitionBy("perfil").orderBy(col("qtd").desc())

top5 = (
  contagem
      .withColumn("rank", row_number().over(window))
      .filter(col("rank") <= 5)
      .orderBy(col("perfil"),col("qtd").desc())
)
top5.display()

In [0]:
# 1. Total de coautores distintos por perfil
total_por_perfil = (
    explodido
      .groupBy("perfil")
      .agg(countDistinct("coautor").alias("total_coautores_distintos"))
      .orderBy(col("total_coautores_distintos").desc())
)

# 2. Trazer para o Pandas para plotar
total_pd = total_por_perfil.toPandas()

# 3. Gráfico de barras
plt.figure(figsize=(10, 5))
sns.barplot(
    data=total_pd,
    x="perfil",
    y="total_coautores_distintos",
    palette="Set2",
    hue="perfil"
)

# 4. Rótulos em cima das barras
for i, v in enumerate(total_pd["total_coautores_distintos"]):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=10)

plt.title("Total de coautores distintos por perfil")
plt.xlabel("Perfil")
plt.ylabel("Total de coautores distintos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:

# Ordem alfabética — determinística e estável
ordem_perfis = sorted(df.select("perfil").distinct().toPandas()["perfil"].tolist())

def plotar_distribuicao(df, coluna, palette, titulo):
    dados = (
        df.groupBy("perfil", coluna)
          .agg(count("*").alias("qtd"))
          .toPandas()
    )
    
    # Ordem das categorias (hue) por volume total decrescente
    hue_order = (
        dados.groupby(coluna)["qtd"].sum()
             .sort_values(ascending=False)
             .index.tolist()
    )
    
    plt.figure(figsize=(12, 5))
    ax = sns.barplot(
        data=dados,
        x="perfil",
        y="qtd",
        hue=coluna,
        order=ordem_perfis,      # <- ordem dos perfis (alfabética)
        hue_order=hue_order,     # <- ordem das categorias (por volume)
        palette=palette
    )
    
    # Rótulos de dados
    for container in ax.containers:
        labels = [
            f"{int(v.get_height()):,}" if v.get_height() > 0 else ""
            for v in container
        ]
        ax.bar_label(container, labels=labels, fontsize=8, padding=2)
    
    plt.title(titulo)
    plt.xlabel("Perfil")
    plt.ylabel("Quantidade de posts")
    plt.xticks(rotation=45)
    plt.legend(title=coluna, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plotar_distribuicao(df, "type", "Set2", "Distribuição de type por perfil")
plotar_distribuicao(df, "productType", "Set3", "Distribuição de productType por perfil")

In [0]:
# drop de colunas que não agregam à análise
df = df.drop('type')

1. Vídeos são o ponto focal da estratégia da maioria dos candidatos no instagram, sobre tudo Renan
2. Lula e Samara investiram no formato carrossel enquanto Renan e Flávio praticamente ignoraram esse formato de conteúdo.
3. Samara Martins tem um perfil mais co-autoral do que focado em suas próprias postagens, o inverso de todos os outros candidatos
4. Samara tem um equilíbrio tem umas estratégia de entre os 3 tipos de postagem, apesar de vídeo ser um dos trabalhos centrais.

### **2.5 Novas Features**

In [0]:
# funções de apoio
def adicionar_dia_semana(df, coluna_ts="timestamp"):
    df = (
        df.withColumn("ts", to_timestamp(col(coluna_ts)))
          .withColumn("_dow_en", date_format(col("ts"), "EEE"))
          .withColumn(
              "dia_semana",
              when(col("_dow_en") == "Mon", "seg")
              .when(col("_dow_en") == "Tue", "ter")
              .when(col("_dow_en") == "Wed", "qua")
              .when(col("_dow_en") == "Thu", "qui")
              .when(col("_dow_en") == "Fri", "sex")
              .when(col("_dow_en") == "Sat", "sáb")
              .when(col("_dow_en") == "Sun", "dom")
          )
          .drop("_dow_en")
    )
    return df

In [0]:
# Criação de features comportamentais derivadas

# Tem mençãos
df = df.withColumn("tem_mencao", when(col("mentions").isNotNull(), True).otherwise(False))

# É criador do post
df = marcar_owner(df, owner_map)

# dia exato da semana
df = adicionar_dia_semana(df)

# turno da postagem (usa 'ts' que é timestamp proper, e hour() para extrair a hora)
df = df.withColumn("turno", when(hour(col("ts")) >= 20, "noite")
                                 .when(hour(col("ts")) >= 13, "tarde")
                                 .when(hour(col("ts")) >= 6, "manha")
                                 .otherwise("madrugada"))

# é fim de semana
df = df.withColumn("fim_semana", when(col("dia_semana") == "sáb", True)
                                         .when(col("dia_semana") == "dom", True)
                                         .otherwise(False))


In [0]:
ESTADOS = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM",
    "Bahia": "BA", "Ceará": "CE", "Distrito Federal": "DF",
    "Espírito Santo": "ES", "Goiás": "GO", "Maranhão": "MA",
    "Mato Grosso do Sul": "MS", "Mato Grosso": "MT",
    "Minas Gerais": "MG",
    "Paraíba": "PB", "Paraná": "PR", "Pernambuco": "PE", "Piauí": "PI",
    "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN", "Rio Grande do Sul": "RS",
    "Rondônia": "RO", "Roraima": "RR", "Santa Catarina": "SC",
    "São Paulo": "SP", "Sergipe": "SE", "Tocantins": "TO"
}

ESTADOS_ORDENADOS = sorted(ESTADOS.items(), key=lambda x: -len(x[0]))

# Contexto que indica uso geográfico do Pará
CONTEXTO_PARA = re.compile(
    r"\b(no|do|ao|pro|pra|para o|para a|estado do|estado do|chegando no|vindo do)\s+Par[áa]\b",
    re.IGNORECASE
)

# "Pará" com acento é sempre estado
PARA_ACENTUADO = re.compile(r"\bPará\b")

def extrair_estados(texto_original):
    """Extrai siglas de estados mencionados no texto."""
    if texto_original is None:
        return []
    
    estados = set()
    texto_restante = texto_original
    
    # Busca por estado
    for nome, uf in ESTADOS_ORDENADOS:
        padrao = rf"\b{re.escape(nome)}\b"
        if re.search(padrao, texto_restante):
            estados.add(uf)
            texto_restante = re.sub(padrao, "", texto_restante)
    
    # Buscar Pará
    if PARA_ACENTUADO.search(texto_restante) or CONTEXTO_PARA.search(texto_restante):
        estados.add("PA")
    
    return sorted(estados)

extrair_udf = udf(extrair_estados, ArrayType(StringType()))
df = df.withColumn("estados_mencionados", extrair_udf(col("caption")))

In [0]:
# para palavras comuns
df = df.withColumn(
    "caption_norm",
    lower(translate(col("caption"),
                    "áàâãäéèêëíìîïóòôõöúùûüç",
                    "aaaaaeeeeiiiiooooouuuuc"))
)

# para siglas críticas
df = df.withColumn(
    "caption_casesensitive",
    translate(
        col("caption"),
        "áàâãäéèêëíìîïóòôõöúùûüç",
        "aaaaaeeeeiiiiooooouuuuc"
    )
)

In [0]:

# Mapas de termos

# --- Termos do PRÓPRIO PARTIDO de cada perfil ---
TERMOS_PARTIDO = {
    "renan":   [r"(?<!a )\bMiss[ãa]o\b"],
    "samara":  [r"\bUP\b", r"\bUnidade\s+Popular\b"],
    "lula":    [r"\bPT\b", r"\bpetista[s]?\b", r"\bpetralha[s]?\b"],
    "flavio":  [r"(?<!a )\bPL\b(?!\s+(do|da|de)\b)"],
    "ronaldo": [r"\bPSD\b"],
}

# Menção a povos indígenas
df = df.withColumn(
    "tem_indigena",
    lower(col("caption")).rlike(
        r"\b(indigena|indigenas|indio|indios|povos originarios|originarios|"
        r"terra indigena|demarcacao|guarani|yanomami|kaingang|terena|"
        r"pataxo|xavante|tupi|aldeia|funai)\b"
    )
)

# Menção ao PRÓPRIO partido
expr_proprio = None
for perfil, regexes in TERMOS_PARTIDO.items():
    regex_unificado = "|".join(regexes)
    cond = (col("perfil") == perfil) & (col("caption_casesensitive").rlike(regex_unificado))
    expr_proprio = (
        when(cond, lit(True)) if expr_proprio is None
        else expr_proprio.when(cond, lit(True))
    )

df = df.withColumn("tem_proprio_partido", expr_proprio.otherwise(lit(False)))

# Menção a OUTRO partido (que não seja o do perfil)
expr_outro = None
for perfil in TERMOS_PARTIDO.keys():
    termos_outros = []
    for p, regexes in TERMOS_PARTIDO.items():
        if p != perfil:
            termos_outros.extend(regexes)
    regex_outros = "|".join(termos_outros)
    cond = (col("perfil") == perfil) & (col("caption_casesensitive").rlike(regex_outros))
    expr_outro = (
        when(cond, lit(True)) if expr_outro is None
        else expr_outro.when(cond, lit(True))
    )

df = df.withColumn("tem_outro_partido", expr_outro.otherwise(lit(False)))

df.select(
    "perfil",
    "tem_indigena",
    "tem_proprio_partido",
    "tem_outro_partido",
    "caption"
).show(10, truncate=False)

def adicionar_qtd_palavras(df, coluna="caption"):
    df = df.withColumn(
        "qtd_palavras",
        when(
            col(coluna).isNull() | (trim(col(coluna)) == ""),
            lit(0)
        ).otherwise(
            size(split(trim(col(coluna)), r"\s+"))
        )
    )
    return df

# tem_pcd
df = df.withColumn(
            "tem_PCD",
             col("caption_norm").rlike(r"\b(pcd|pessoa com deficiência|pessoa com deficiencia|"
                              r"deficiente|deficientes|deficiência|deficiencia|"
                              r"cadeirante|cadeirantes|autista|autistas|"
                              r"neurodivergente|neurodivergentes|"
                              r"acessibilidade|inclusão|inclusao)\b")
        ) 

 # tem lgbt
df = df.withColumn(
            "tem_LGBT",
             col("caption_norm").rlike(r"\b(lgbt|lgbtqia|lgbtqia\+|lgbtq|"
                              r"gay|gays|lésbica|lesbica|lésbicas|lesbicas|"
                              r"bissexual|bissexuais|travesti|travestis|"
                              r"trans|transgênero|transgenero|transgêneres|"
                              r"homossexual|homossexuais|"
                              r"homofobia|transfobia|"
                              r"orgulho gay|parada gay|parada lgbt|causa lgbt)\b")
)


df = adicionar_qtd_palavras(df)

In [0]:

# ------ Colunas booleanas

# Horário: XX:XX ou Xh / XXh
df = df.withColumn(
    "tem_horario",
    col("caption_norm").rlike(r"\b\d{1,2}:\d{2}\b") |
    col("caption_norm").rlike(r"\b\d{1,2}h\b")
)

# Mulher
df = df.withColumn(
    "tem_mulher",
    col("caption_norm").rlike(r"\bmulher(es)?\b")
)

# Negro / negra / racismo
df = df.withColumn(
    "tem_negros",
    col("caption_norm").rlike(r"\b(negro|negra|negros|negras|racismo)\b")
)

# 6x1
df = df.withColumn(
    "tem_6x1",
    col("caption_norm").rlike(r"\b6\s*x\s*1\b")
)

# Saúde
df = df.withColumn(
    "tem_saude",
    col("caption_norm").rlike(r"\bsa[uú]de\b")
)

# Educação
df = df.withColumn(
    "tem_educacao",
    col("caption_norm").rlike(r"\beduca[cç][aã]o\b")
)

# segurança
df = df.withColumn(
    "tem_seguranca",
    col("caption_norm").rlike(r"\b(seguran[cç]a|seguran[cç]a pu[úu]blica|seguran[cç]a cidada|seguran[cç]a social)\b")
)

df = df.drop("caption_norm", "estado_por_sigla","caption_casesensitive")

df.display()

In [0]:
# atualizar campos booleanos
campos_booleanos.remove("isCommentsDisabled")
campos_booleanos.remove("paidPartnership")

campos_booleanos_novos = [
    "fim_semana",
    "tem_mencao",
    "LikesCount_unknown",
    "tem_horario",
    "tem_mulher",
    "tem_negros",
    "tem_6x1",
    "tem_saude",
    "tem_educacao",
    "tem_seguranca",
    "tem_indigena",
    "tem_proprio_partido",
    "tem_outro_partido"
]

campos_numericos.append("qtd_palavras")

campos_booleanos.extend(campos_booleanos_novos)
# ---------------------------------------------------------

colunas_validas = list(dict.fromkeys(campos_booleanos + campos_numericos))
df_pd = df.select("perfil", *colunas_validas).toPandas()

for c in df_pd.columns:
    if df_pd[c].dtype == bool:
        df_pd[c] = df_pd[c].astype(int)

# Lista de perfis
perfis = sorted(df_pd["perfil"].dropna().unique().tolist())

# Grade de subplots
n = len(perfis)
ncols = 2
nrows = (n + 1) // 2 

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 8 * nrows))
axes = axes.flatten()

for i, perfil in enumerate(perfis):
    ax = axes[i]
    
    subset = df_pd[df_pd["perfil"] == perfil].drop(columns="perfil")
    subset = subset.loc[:, subset.nunique() > 1]
    
    if subset.shape[1] < 2:
        ax.set_title(f"{perfil} (sem variáveis suficientes)")
        ax.axis("off")
        continue
    
    # Matriz de correlação
    corr = subset.corr(method="spearman", numeric_only=True)
    
    # Máscara do triângulo superior
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.7},
        ax=ax,
        annot_kws={"size": 8}
    )
    ax.set_title(f"{perfil} (n={len(subset)})", fontsize=12)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)

# Desliga eixos vazios
for j in range(n, len(axes)):
    axes[j].axis("off")

plt.suptitle("Matriz de correlação por perfil (Spearman)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### **Percepções da EDA**

1. Comportamento dos candidatos quanto ao conteúdo publicado
    
   **Tipo de conteúdo**
    - Flávio, Ronaldo: Priorizam videos curtos em volume rasoável.
    - Lula: Diversifica o tipo de postagem ainda priorizando vídeos curtos
    - Renan: Prioriza grande volume de videos, um pouco mais longos, mas ainda curtos, e ignora carrosseis
    - Samara: Foca na diversificação dos tipos de postagem 

    **Colaboração**
    - Lula e Flávio: Colaboram com algumas pessoas, mas não é um valor expressivo.
    - Flávio, Renan e Ronaldo: O foco não é a colaboração, é a construção da candidatura de forma individual.
    - Samara: Movimento oposto ao dos candidatos, tem mais postagens na qual é co-autora do que o contrário. Tem um perfil de envolver mais outras instâncias do seu partido na sua candidatura.
    - Cos candidatos, exceto samara, tendem a ter seus coautores com perfil verificado.
    
    **Flexibilidade**
    - Não foram percebidas mudanças drásticas na abordagem ao longo do tempo com essa análise inicial

    **Temas**
    - Não aborda 6x1: Flávio
    - Não aborda Negros ou racismo: Renan e Flávio
    - Não aborda saúde: Renan
    - Não aborda educação: nenhum
    - Não aborda educação: nenhum

2. Performance do conteúdo

    **Curtidas**
    - Flávio
        _**Geral**: Seu perfil tem muitos likes para uma quantidade de postagens proporcionalmente menor que a de outros candidatos, tem a performance mais alta no instagram dentre todos._

      - Duração de video e quantidade de imagens no carrossem nao tem influência nas interações.
      - Vídeos mais longos tem uma leve tendência a ter mais views
      - No período da campanha teve um grande crescimento nas interações
      - Suas postagens não tratam de temas sociais e isso nao parece afetar negativamente a performance, a ausência parece intencional.

    - Lula
        _**Geral**: Tem uma mediana de curtidas alta mas inconstante durante a pré campanha e com um leve crescimento linear ao longo da campanha._

      - Tem a proporção de comentários em relação à likes maior que os outros candidatos.
      - Quanto mais longo seu carrossel menor a quantidade de comentários e curtidas.
      - Vídeos mais longos tem mais comentários e uma leve tendência a ter mais curtidas.
      - Seus posts que mencionam outros partidos tem uma leve tendência a ter mais likes
      - Posts com horário tem uma tendência a ter menos likes e comentários.
      - Posts que mencionam o próprio partido tem uma tendencia a mais likes e comentários.
      - Videos mais longos tem forte tendência a ter mais visualizações.

    - Renan

         _**Geral**: Tem uma mediana de curtidas acima de Lula na maior parte do período e abaixo de Flávio_

        - Quanto mais longo seu carrossel existe uma leve tendência a ter menos  comentários e curtidas.
        - Vídeos mais longos tem uma tendência a ter mais likes e comentários.
        - Posts com horário tem uma leve tendência a ter menos likes, curtidas e visualiazações.

    - Ronaldo

         _**Geral**: Muito abaixo da performance dos 3 anteriores mas ainda sim, mais que o dobro que a de Samara (mesmo ela tendo mais postagens)._

      - Quanto mais longo seu carrossel menor a quantidade de comentários e curtidas.
      - Vídeos mais longos tem uma tendência a ter mais likes e comentários.
      - Posts com horário tem uma tendência a ter menos likes.
      - Existe uma tendência em curtidas, comentarios e visualizações para o assunto "segurança"
      - Posts que mencionam o próprio partido tem uma tendencia a menos likes e comentários.


    - Samara

       _**Geral**: Performance muito abaixo de suas contrapartidas tanto em likes, visualizações e comentários._

      - Tem mais curtidas e comentários em postagens das quais é autora.
      - Quanto mais longo seu carrossel menor a quantidade de comentários mas esse tamanho não afeta a curtida.
      - Vídeos mais longos tem mais comentários e uma leve tendência a ter mais curtidas.
      - Seus posts com músicas tem mais comentários e uma leve tendência a ter mais likes.
      - Posts com horário tem uma tendência a ter menos likes e comentários.
      - Posts que mencionam o próprio partido tem uma tendencia a menos likes e comentários.


3. Padrões a serem investigados


In [0]:
# ordenação das semanas
def adicionar_numero_semana(df, coluna_ts="timestamp", semana_ref="2026-02-01"):
    df = (
        df.withColumn("ts", to_timestamp(col(coluna_ts)))
          .withColumn(
              "num_semana",
              floor(datediff(col("ts"), lit(semana_ref)) / 7).cast("int")
          )
    )
    return df

# Campos booleanos em int
def booleanos_para_int(df):
    """Converte colunas booleanas para int (1/0)."""
    for campo, tipo in df.dtypes:
        if tipo == "boolean":
            df = df.withColumn(campo, col(campo).cast("int"))
    return df

from pyspark.sql.functions import col, when, size, element_at, lit

def adicionar_coluna_estado(df, coluna_lista="estados_mencionados"):
    df = df.withColumn(
        "estado",
        when(size(col(coluna_lista)) == 0, lit("nenhum"))
        .when(size(col(coluna_lista)) == 1, element_at(col(coluna_lista), 1))
        .otherwise(array_join(col(coluna_lista), ", "))
    )
    return df

def adicionar_região_por_estado(df):
    df = df.withColumn(
        "regiao",
        when(col("estado").isin(["AC", "AM", "PA", "RO", "RR", "TO"]), lit("NORTE"))
        .when(col("estado").isin(["AP", "MA", "PI", "RN", "CE", "PB", "PE", "AL", "SE", "BA"]), lit("NORDESTE"))
        .when(col("estado").isin(["MT", "MS", "GO"]), lit("CENTRO-OESTE"))
        .when(col("estado").isin(["ES", "MG", "RJ", "SP"]), lit("SUDESTE"))
        .when(col("estado").isin(["DF", "SC", "RS"]), lit("SUL"))
        .otherwise(lit("OUTROS"))
    )
    return df

df = df.withColumn(
    "fase_campanha",
    when(to_date("timestamp") < "2026-07-21", "Pré-campanha")
    .when((to_date("timestamp") >= "2026-07-22") & (to_date("timestamp") <= "2026-08-04"), "Lançamento candidatura")
    .when((to_date("timestamp") >= "2026-08-05") & (to_date("timestamp") <= "2026-10-03"), "Período eleitoral")
    .otherwise("Fora do período")
)



df = adicionar_numero_semana(df)
df = adicionar_coluna_estado(df)
df = adicionar_região_por_estado(df)
df = booleanos_para_int(df)
    

In [0]:

df.display()


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.2026eleicoes;

In [0]:
# Salva o DataFrame final como uma tabela no Unity Catalog
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.2026eleicoes.pipeline_2026")